In [2]:
from ultralytics import YOLO
import pandas as pd

def eval_yolo_seg(model_path, data_yaml, split="test", imgsz=640, batch=16, plots=True):
    model = YOLO(model_path)

    try:
        metrics = model.val(
            data=data_yaml,
            split=split,
            imgsz=imgsz,
            batch=batch,
            plots=plots
        )
    except Exception as e:
        print("Ошибка во время val():")
        print(type(e).__name__, e)
        print("Скорее всего проблема в датасете/разметке, а не в коде метрик.")
        return None, None

    rows = [{
        "class": "all",
        "box_P": float(metrics.box.mp),
        "box_R": float(metrics.box.mr),
        "box_mAP50": float(metrics.box.map50),
        "box_mAP50-95": float(metrics.box.map),
        "mask_P": float(metrics.seg.mp),
        "mask_R": float(metrics.seg.mr),
        "mask_mAP50": float(metrics.seg.map50),
        "mask_mAP50-95": float(metrics.seg.map),
    }]

    for i, cls_name in metrics.names.items():
        row = {"class": cls_name}
        if hasattr(metrics.box, "class_result"):
            p, r, ap50, ap = metrics.box.class_result(i)
            row.update({
                "box_P": float(p),
                "box_R": float(r),
                "box_mAP50": float(ap50),
                "box_mAP50-95": float(ap),
            })
        if hasattr(metrics.seg, "class_result"):
            p, r, ap50, ap = metrics.seg.class_result(i)
            row.update({
                "mask_P": float(p),
                "mask_R": float(r),
                "mask_mAP50": float(ap50),
                "mask_mAP50-95": float(ap),
            })
        rows.append(row)

    df = pd.DataFrame(rows)
    return metrics, df

In [12]:
model = YOLO('/Users/user/Downloads/yolo_seg.pt')
metrics = model.val(data='/Users/user/Downloads/yolo_seg_dataset_final_4/data.yaml', split='test')

Ultralytics 8.4.41 🚀 Python-3.12.2 torch-2.7.0 CPU (Apple M1)
YOLO11m-seg summary (fused): 139 layers, 22,356,900 parameters, 0 gradients, 123.1 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 610.5±387.2 MB/s, size: 249.0 KB)
val: Scanning /Users/user/Downloads/yolo_seg_dataset_final_4/test/labels... 520 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 520/520 1.5Kit/s 0.4s<0.1s
val: New cache created: /Users/user/Downloads/yolo_seg_dataset_final_4/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 9.9s/it 5:27<10.0s
                   all        520       1546       0.59      0.514      0.544       0.46      0.591      0.518      0.547      0.434
           bottle-blue         87        104      0.693      0.548      0.567      0.455      0.692      0.548      0.572      0.424
          bottle-green         65         74      0.674        0.5      0

In [6]:
model = YOLO('/Users/user/Downloads/SEG_CAP_BEST.pt')
metrics = model.val(data='/Users/user/Downloads/yolo_seg_dataset_CAP 3/data.yaml', split='test')

Ultralytics 8.4.41 🚀 Python-3.12.2 torch-2.7.0 CPU (Apple M1)
YOLO11m-seg summary (fused): 139 layers, 22,343,022 parameters, 0 gradients, 123.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 622.2±314.8 MB/s, size: 195.1 KB)
val: Scanning /Users/user/Downloads/yolo_seg_dataset_CAP 3/test/labels.cache... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 34.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 8.9s/it 1:02<11.5ss
                   all         99       2127      0.819      0.824      0.836      0.742      0.816      0.822      0.834      0.485
                  blue         95        455      0.981      0.918      0.983      0.889      0.981      0.918      0.983      0.588
                 brown         60        118      0.956      0.983      0.963      0.792      0.948      0.975      0.954      0.519
                 green         91